# Scrape antifakenewscenter.com – #มิจฉาชีพ news

Keep only posts tagged **#มิจฉาชีพ** together with **#อาชญากรรมออนไลน์** and/or **#ข่าวปลอม**.

The site is WordPress, so instead of parsing the tag page HTML we read the same posts of the
`/tag/มิจฉาชีพ/` archive through its public REST API (`/wp-json/wp/v2/posts?tags=<id>`). Each post
already carries its tag IDs, so no per-post page fetch is needed.

Output columns: `post_id` (the site's WordPress post ID), `date` (publish date, `DD/MM/YYYY` in the
CSV and a datetime column in the dataframe), `hashtag`, `title`, `content`, `Channel`, `ORG` and `scam_type`.

`Channel` and `ORG` are labelled with keyword rules on the title and content, so they are a first
pass, not ground truth:
- `Channel`: `call` if the scammer phones the victim, otherwise `text`.
- `ORG`: the Thai organization(s) the scam is about, from the reference list built further
  down (see "Organization column"). Empty when no listed organization is found.
- `scam_type`: one of the five Callas scam types, labelled by an LLM (see "Scam type column").

In [ ]:
import html
import io
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

API = "https://www.antifakenewscenter.com/wp-json/wp/v2"
HEADERS = {"User-Agent": "Mozilla/5.0 (research scraper; Callas project)"}
DELAY = 1.0  # seconds between requests

BASE_TAG = "มิจฉาชีพ"
TARGET_TAGS = ["อาชญากรรมออนไลน์", "ข่าวปลอม"]
OUT_PATH = Path("data/antifakenews_scam.csv")

In [14]:
def get_json(path, params, retries=3):
    """GET an API endpoint with retries; returns (json, response headers)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{API}/{path}", params=params, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json(), r.headers
        except requests.RequestException:
            if attempt == retries:
                raise
            time.sleep(2 * attempt)


def find_tag_id(name):
    """Resolve a tag name (with or without a leading '#') to its WordPress tag ID."""
    tags, _ = get_json("tags", {"search": name, "per_page": 100, "_fields": "id,name,count"})
    exact = [t for t in tags if t["name"].lstrip("#").strip() == name]
    if not exact:
        raise ValueError(f"tag not found: {name}")
    return max(exact, key=lambda t: t["count"])["id"]


tag_ids = {name: find_tag_id(name) for name in [BASE_TAG, *TARGET_TAGS]}
tag_ids

{'มิจฉาชีพ': 88, 'อาชญากรรมออนไลน์': 5713, 'ข่าวปลอม': 11}

In [ ]:
def fetch_posts(tag_id):
    """All posts of a tag (same set as the /tag/<slug>/ archive pages)."""
    posts, page, total_pages = [], 1, 1
    while page <= total_pages:
        batch, headers = get_json("posts", {"tags": tag_id, "per_page": 100, "page": page,
                                            "_fields": "id,date,title,content,tags"})
        total_pages = int(headers["X-WP-TotalPages"])
        posts += batch
        print(f"page {page}/{total_pages}: {len(posts)}/{headers['X-WP-Total']} posts")
        page += 1
        time.sleep(DELAY)
    return posts


raw_posts = fetch_posts(tag_ids[BASE_TAG])

In [ ]:
FOOTER_HEADING = "ช่องทางการติดตามและแจ้งเบาะแสข่าวปลอม"
FOOTER_START = "LINE : @antifakenewscenter"  # the site's contact block closes every post from here on

# The scammer phones the victim -> "call", anything else -> "text".
# Only phrases about being called count; a bare "โทรศัพท์" or a report hotline does not.
CALL = (r"คอลเซ็นเตอร์|ทางโทรศัพท์|ผ่านโทรศัพท์|(?i:\bcall\s*center\b)|รับสาย|ได้รับสาย"
        r"|โทรศัพท์(?:มา|หา|ติดต่อ|สอบ|แจ้ง|ถาม|ติดตาม)"
        r"|โทร(?:มา|หา|ไป|แจ้ง|ติดต่อ|สอบ|ถาม|ตรวจสอบ|หลอก|ขู่|เข้า)")
HOTLINE = r"(?i:call\s*center)\s*\d{3,4}|สายด่วน\s*\d+"  # e.g. "Call Center 1284", not a scam call


def clean_html(fragment):
    soup = BeautifulSoup(fragment, "html.parser")
    for tag in soup(["script", "style", "iframe", "figure", "img"]):
        tag.decompose()
    lines = (re.sub(r"[ \t]+", " ", line).strip() for line in soup.get_text("\n").replace("\xa0", " ").splitlines())
    lines = [line for line in lines if line]
    # drop the contact footer (LINE, website, Facebook, X, TikTok, IG) and its heading
    for i, line in enumerate(lines):
        if line.startswith(FOOTER_START):
            lines = lines[:i]
            if lines and lines[-1] == FOOTER_HEADING:
                lines.pop()
            break
    return "\n".join(lines)


def label_channel(text):
    return "call" if re.search(CALL, re.sub(HOTLINE, "", text)) else "text"


rows = []
for p in raw_posts:  # all fetched under #มิจฉาชีพ; keep those that also carry a target hashtag
    matched = [name for name in TARGET_TAGS if tag_ids[name] in p["tags"]]
    if matched:
        rows.append({
            "post_id": p["id"],
            "date": p["date"],
            "hashtag": ", ".join(matched),  # both tags -> one row, e.g. "อาชญากรรมออนไลน์, ข่าวปลอม"
            "title": html.unescape(BeautifulSoup(p["title"]["rendered"], "html.parser").get_text()).strip(),
            "content": clean_html(p["content"]["rendered"]),
        })

df = pd.DataFrame(rows)
# real datetime64 column, so it sorts chronologically (a DD/MM/YYYY string would sort by day first)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["Channel"] = (df["title"] + "\n" + df["content"]).map(label_channel)
df = df.sort_values("date", ascending=False, ignore_index=True)
df

## Organization column

The `ORG` column names the Thai organization each post says the scammer pretended to be,
taken from a reference list of organizations with their **full names and abbreviations**:

| Source | What it gives |
|---|---|
| [DGA open data, "รายชื่อหน่วยงานของรัฐส่วนกลางระดับกรมหรือเทียบเท่า"](https://catalog-dga.data.go.th/dataset/di-open1-01) | 413 central agencies (departments, state enterprises, independent agencies) with Thai name, English name and Thai abbreviation. Only a few rows carry an English abbreviation. Each ministry is added too, as its own row. |
| [Thai Wikipedia, รายชื่อธนาคารในประเทศไทย](https://th.wikipedia.org/wiki/รายชื่อธนาคารในประเทศไทย) | Banks currently operating, with the SET ticker or the state-bank abbreviation (BBL, KTB, SCB, GSB ...). |
| [Wikidata](https://www.wikidata.org) + English Wikipedia | English acronyms. The Thai name is looked up on Wikidata, which links to the English article, whose first sentence usually gives the acronym, e.g. "The Royal Thai Police (RTP) ...". An acronym is kept only if the article is about Thailand and the acronym fits the English name, because the bracket sometimes holds the parent ministry instead. |

The merged list is saved to `data/thai_organizations.csv`. Most organizations still have no English
abbreviation (80 of 451 do), because many have no English Wikipedia article or the article gives none.

A post is matched on any alias of an
organization (full name, abbreviation, `ธ.`/`แบงก์` + bank name, English name) after removing spaces,
because Thai spacing is irregular. The title is searched first. If it names no organization, the most
mentioned one in the content is used, ignoring the police, since almost every post tells the reader to
report to them. Limits: private companies (HomePro, Lotus's, Singha ...) and local bodies are not in
the list, so those posts get an empty `ORG`, and a match means the post *mentions* the
organization, not strictly that the scammer impersonated it.

In [ ]:
DGA_CSV = ("https://catalog-dga.data.go.th/dataset/af5f5723-0a54-47be-900f-19420585cc56/resource/"
           "64052093-d954-44f4-9993-d660f9fa245a/download/.csv")
BANKS_URL = "https://th.wikipedia.org/wiki/รายชื่อธนาคารในประเทศไทย"
ORG_PATH = Path("data/thai_organizations.csv")

has_thai = lambda s: bool(re.search(r"[฀-๿]", s))


def download(url, timeout=30):
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r


def load_dga_orgs():
    """Agencies, plus each ministry (the list only has its permanent-secretary office)."""
    d = pd.read_csv(io.BytesIO(download(DGA_CSV, timeout=60).content), encoding="utf-8-sig")
    # OrgNameAbbr mixes Thai and English abbreviations, e.g. "BOI, สกท."
    abbrs = d["OrgNameAbbr"].fillna("").map(lambda s: [a.strip() for a in s.split(",") if a.strip()])
    agencies = pd.DataFrame({
        "name_th": d["OrgNameThai"].str.strip(),
        "name_en": d["OrgNameEnglish"].astype(str).str.strip(),
        "abbr_th": abbrs.map(lambda a: "|".join(x for x in a if has_thai(x))),
        "abbr_en": abbrs.map(lambda a: "|".join(x for x in a if not has_thai(x))),
        "type": d["MajorCategoryName_byLaw"],
        "source": "DGA open data",
    })
    offices = d[d["OrgNameThai"].str.startswith("สำนักงานปลัดกระทรวง")].drop_duplicates("MinistryNameThai")
    office_abbr = dict(zip(offices["MinistryNameThai"], offices["OrgNameAbbr"].astype(str).str.strip()))
    ministries = pd.DataFrame([
        {"name_th": m.strip(), "name_en": "", "abbr_en": "", "type": "กระทรวง", "source": "DGA open data (ministry)",
         # สป.กค. (office of the ministry) -> กค. (the ministry)
         "abbr_th": office_abbr[m][3:].lstrip() if office_abbr.get(m, "").startswith("สป.") else ""}
        for m in sorted(d["MinistryNameThai"].dropna().unique()) if m.startswith("กระทรวง")
    ])
    return pd.concat([agencies, ministries])


def load_bank_orgs():
    rows = []
    soup = BeautifulSoup(download(BANKS_URL).text, "html.parser")
    for table in soup.select("table.wikitable")[:5]:  # banks still operating
        for tr in table.find_all("tr")[1:]:
            c = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
            if len(c) >= 5 and c[1].startswith("ธนาคาร"):
                m = re.search(r"(?:SET\s*:|ตัวอักษรย่อ\s*:)\s*([A-Z]+)", c[3])  # "SET: BBL" / "ตัวอักษรย่อ: GSB"
                rows.append({"name_th": c[1], "name_en": "", "abbr_th": "", "abbr_en": m.group(1) if m else "",
                             "type": "ธนาคาร", "source": "Thai Wikipedia"})
    return pd.DataFrame(rows)


# --- English acronyms: Thai name -> Wikidata item -> English Wikipedia article -> "(RTP)" in its first sentence
WIKI_HEADERS = {"User-Agent": "CallasResearch/0.1 (https://github.com/ZenithGeeks/Callas; scam dataset)"}
ACRONYM_IN_INTRO = re.compile(r"\((?:abbreviated(?: as)?:?\s*|abbr\.?:?\s*)?([A-Z][A-Za-z]*[A-Z]|[A-Z](?:\.[A-Z])+\.?)\)"
                              r"|abbreviated(?: as)? ([A-Z]{2,})")


def wiki_get(url, **params):
    """Wikimedia APIs ask for a descriptive User-Agent and sometimes rate limit, so retry."""
    for attempt in range(5):
        r = requests.get(url, params={**params, "format": "json"}, headers=WIKI_HEADERS, timeout=60)
        if r.ok and r.headers.get("content-type", "").startswith("application/json"):
            return r.json()
        time.sleep(3 * (attempt + 1))
    r.raise_for_status()
    raise RuntimeError(f"no JSON from {url}")


def acronym_fits(acronym, name):
    """RTP fits "Royal Thai Police": same first letter, letters in order, last letter starts a word."""
    words = re.sub(r"\([^)]*\)", "", name).split()
    words = words[1:] if words and words[0].lower() == "the" else words
    letters, text = re.sub(r"[^A-Za-z]", "", acronym).upper(), " ".join(words).upper()
    if not words or not letters or letters[0] != text[0] or letters[-1] not in {w[0].upper() for w in words}:
        return False
    rest = iter(text)
    return all(ch in rest for ch in letters)


def load_wikipedia_acronyms(orgs):
    """English acronyms of the organizations whose English Wikipedia article gives one."""
    item_names = {}  # Wikidata item -> Thai names with that label
    for i in range(0, len(orgs), 100):
        values = " ".join(f'"{n}"@th' for n in orgs["name_th"].iloc[i:i + 100])
        r = requests.post("https://query.wikidata.org/sparql", headers={**WIKI_HEADERS, "Accept": "application/json"},
                          data={"query": f"SELECT ?item ?th WHERE {{ VALUES ?th {{ {values} }} ?item rdfs:label ?th }}"},
                          timeout=120)
        r.raise_for_status()
        for b in r.json()["results"]["bindings"]:
            item_names.setdefault(b["item"]["value"].rsplit("/", 1)[-1], []).append(b["th"]["value"])

    items, article = list(item_names), {}  # Wikidata item -> English article title
    for i in range(0, len(items), 50):
        entities = wiki_get("https://www.wikidata.org/w/api.php", action="wbgetentities", ids="|".join(items[i:i + 50]),
                            props="sitelinks", sitefilter="enwiki")["entities"]
        article |= {q: e["sitelinks"]["enwiki"]["title"] for q, e in entities.items() if "enwiki" in e.get("sitelinks", {})}

    titles, intro = sorted(set(article.values())), {}  # article title -> plain-text intro
    for i in range(0, len(titles), 20):
        q = wiki_get("https://en.wikipedia.org/w/api.php", action="query", prop="extracts", exintro=1, explaintext=1,
                     exlimit=20, redirects=1, titles="|".join(titles[i:i + 20]))["query"]
        renamed = {x["from"]: x["to"] for x in q.get("normalized", []) + q.get("redirects", [])}
        text = {p["title"]: p.get("extract", "") for p in q["pages"].values()}
        for t in titles[i:i + 20]:
            intro[t] = text.get(renamed.get(renamed.get(t, t), renamed.get(t, t)), "")

    name_en = dict(zip(orgs["name_th"], orgs["name_en"]))
    rows = []
    for item, title in article.items():
        m = ACRONYM_IN_INTRO.search(intro[title][:350])
        if not m or "Thai" not in intro[title][:600]:  # no acronym, or an article about another country
            continue
        acronym = m.group(1) or m.group(2)
        for th in item_names[item]:
            # the bracket sometimes holds the parent ministry (e.g. MoNRE), so the acronym must fit the name
            if any(acronym_fits(acronym, n) for n in (name_en.get(th, ""), title) if n):
                rows.append({"name_th": th, "abbr_en": acronym, "source": "English Wikipedia"})
    return pd.DataFrame(rows)


first_filled = lambda s: next((v for v in s if v), "")
join_unique = lambda s: "|".join(dict.fromkeys(a for v in s for a in v.split("|") if a))


def merge_orgs(frames):
    """One row per Thai name; abbreviations from every source are combined."""
    return (pd.concat(frames, ignore_index=True).fillna("")
            .groupby("name_th", sort=False, as_index=False)
            .agg(name_en=("name_en", first_filled), abbr_th=("abbr_th", join_unique),
                 abbr_en=("abbr_en", join_unique), type=("type", "first"), source=("source", "first")))


orgs = merge_orgs([load_dga_orgs(), load_bank_orgs()])  # state banks are in both sources
orgs = merge_orgs([orgs, load_wikipedia_acronyms(orgs)])
ORG_PATH.parent.mkdir(parents=True, exist_ok=True)
orgs.to_csv(ORG_PATH, index=False, encoding="utf-8-sig")
print(f"{len(orgs)} organizations -> {ORG_PATH}; {(orgs['abbr_en'] != '').sum()} with an English abbreviation")
orgs["type"].value_counts()

In [ ]:
def squash(text):
    """Drop Thai spacing (it is irregular) but keep the gap between two Latin words."""
    text = text.replace("​", "")

    def gap(m):
        before = text[m.start() - 1] if m.start() else ""
        after = text[m.end()] if m.end() < len(text) else ""
        return " " if re.match(r"[A-Za-z0-9]", before) and re.match(r"[A-Za-z0-9]", after) else ""

    return re.sub(r"\s+", gap, text)


# colloquial names the lists do not carry: canonical Thai name -> extra aliases
MANUAL_ALIASES = {
    "สำนักงานตำรวจแห่งชาติ": ["ตำรวจ", "ตร."],
    "กรมสอบสวนคดีพิเศษ": ["ดีเอสไอ"],
    "ธนาคารแห่งประเทศไทย": ["แบงก์ชาติ"],
    "ธนาคารทหารไทยธนชาต": ["ทีทีบี"],
}
# ordinary words that must not stand in for an organization on their own
AMBIGUOUS_BANK_BARE = {"กรุงเทพ", "ไทย", "เครดิต"}  # keep "ธนาคาร"/"ธ." in front
GENERIC_COMPANY_BASE = {"ขนส่ง", "การบิน"}          # "บริษัท ขนส่ง จำกัด" minus บริษัท/จำกัด
BLOCKED_ALIASES = {"SME"}                           # loan product word, not the bank
IGNORE_IN_CONTENT = {"สำนักงานตำรวจแห่งชาติ"}        # posts end with "report to the police"


def org_aliases(row):
    name = row.name_th
    base = re.sub(r"\s*\((?:มหาชน|องค์การมหาชน)\)|\s*จำกัด|^บริษัท\s*", "", name).strip()
    out = {name, *MANUAL_ALIASES.get(name, []), *filter(None, row.abbr_en.split("|"))}
    if len(squash(base)) >= 6 and base not in GENERIC_COMPANY_BASE:
        out.add(base)
    if base.startswith("ธนาคาร"):
        rest = base[len("ธนาคาร"):].strip()
        out |= {"ธ." + rest, "แบงก์" + rest}
        if len(rest) >= 4 and rest not in AMBIGUOUS_BANK_BARE:
            out.add(rest)
    for a in filter(None, row.abbr_th.split("|")):
        out |= {a, re.sub(r"^สำนักงาน\s*", "", a)}  # "สำนักงาน ก.ล.ต." is also written "ก.ล.ต."
    # Thai abbreviations need >= 4 characters incl. dots, English ones >= 3, or they hit ordinary words
    return {a for a in out if a not in BLOCKED_ALIASES and len(squash(a)) >= (4 if has_thai(a) else 3)}


ALIAS_TO_ORG = {}
for row in orgs.itertuples():
    for alias in org_aliases(row):
        ALIAS_TO_ORG.setdefault(squash(alias), row.name_th)  # on a clash the first listed organization wins

# one pattern, longest alias first so the longest name wins; English aliases must be whole words
ALIAS_RE = re.compile("|".join(
    re.escape(a) if has_thai(a) else rf"(?<![A-Za-z]){re.escape(a)}(?![A-Za-z])"
    for a in sorted(ALIAS_TO_ORG, key=len, reverse=True)
))


def org_mentions(text):
    """Organizations named in the text, in order of appearance."""
    return [ALIAS_TO_ORG[m.group()] for m in ALIAS_RE.finditer(squash(text))]


def label_organization(title, content):
    in_title = dict.fromkeys(org_mentions(title))  # unique, in order
    if in_title:
        return ", ".join(in_title)
    counts = Counter(org for org in org_mentions(content) if org not in IGNORE_IN_CONTENT)
    return counts.most_common(1)[0][0] if counts else ""


df["ORG"] = [label_organization(t, c) for t, c in zip(df["title"], df["content"])]
labelled = df.loc[df["ORG"] != "", "ORG"]
print(f"{len(labelled)}/{len(df)} posts have an organization")
labelled.str.split(", ").explode().value_counts().head(20)

## Scam type column

`scam_type` puts each post into one of the five Callas scam types (`fake_authority`, `romance`,
`investment`, `phishing_otp`, `lottery`) with an LLM. The prompt is `prompts/classify_scam_type.txt`:
wider definitions of the five types plus a tie-break order for posts that fit several. The call goes
through `llm_client.py`, a small client for any OpenAI-compatible server, kept apart from the app's
`modules/llm_serving`. Set the server in `.env` (copy `.env.example`): `LLM_BASE_URL`, `LLM_MODEL`.

Answers are kept in `data/scam_type_labels.csv` with a hash of the prompt, so a re-run only asks
about new posts, and editing the prompt asks again. An answer that is not exactly one of the five
types becomes `unknown`. If the server is down, the posts not yet labelled get an empty `scam_type`.

In [ ]:
from llm_client import LLMUnavailable, ScamTypeLLM

LABEL_CACHE = Path("data/scam_type_labels.csv")
llm = ScamTypeLLM()

cache = (pd.read_csv(LABEL_CACHE, encoding="utf-8-sig", keep_default_na=False) if LABEL_CACHE.exists()
         else pd.DataFrame(columns=["post_id", "prompt", "raw", "scam_type"]))
cache = cache[cache["prompt"] == llm.prompt_version]  # answers from an older prompt are asked again
todo = df[~df["post_id"].isin(cache["post_id"])]

new = []
try:
    for i, post in enumerate(todo.itertuples(), 1):
        scam_type, raw = llm.classify(f"{post.title}\n{post.content}")
        new.append({"post_id": post.post_id, "prompt": llm.prompt_version, "raw": raw, "scam_type": scam_type})
        if i % 25 == 0 or i == len(todo):
            print(f"{i}/{len(todo)} posts labelled")
except LLMUnavailable as e:
    print(f"LLM server unavailable, {len(todo) - len(new)} posts left without a scam type: {e}")

cache = pd.concat([cache, pd.DataFrame(new)], ignore_index=True)
cache.to_csv(LABEL_CACHE, index=False, encoding="utf-8-sig")
df["scam_type"] = df["post_id"].map(dict(zip(cache["post_id"], cache["scam_type"]))).fillna("")
print(f"prompt {llm.prompt_version}, model {llm.model or '?'} at {llm.base_url}")
df["scam_type"].value_counts()

In [ ]:
print(f"{len(raw_posts)} posts under #{BASE_TAG} -> {len(df)} kept")
display(df["hashtag"].value_counts())
df["Channel"].value_counts()

In [ ]:
assert df["post_id"].is_unique
assert df["date"].notna().all()
assert (df["content"].str.len() > 0).all()
assert df["hashtag"].str.contains("|".join(TARGET_TAGS)).all()
assert not df["content"].str.contains(FOOTER_START, regex=False).any()

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
# the date column is written as DD/MM/YYYY
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig", date_format="%d/%m/%Y")

# read it back with the same format so the column is datetime again and sortable
pd.read_csv(OUT_PATH, parse_dates=["date"], date_format="%d/%m/%Y").head()